Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise 3.3: Table Modeling and Write Optimization

Building on E1.2's partitioning and sort order, this exercise focuses on **write-time optimization** — specifically how Spark's **distribution modes** affect file layout and write performance. Distribution modes are the main new concept; the partition/sort experiments re-validate E1.2 with distribution modes in the mix.

- **Distribution modes** — how Spark shuffles data before writing (main focus)
- **Partition strategies** — quick comparison + write-time cost
- **Sort orders** — write-cost analysis

> Timings illustrate real-world tradeoffs. Absolute numbers vary by machine; production clusters show bigger gaps.

⚠️ **Spark Connect**: one Spark server in the background, notebooks are thin clients. If `ConnectionRefusedError` appears, check `docker logs jupyter-spark` or restart with `docker compose restart jupyter`.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("TableModeling") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.modeling")
print("Namespace 'modeling' created!")

## Download NYC Taxi Data

Yellow Taxi trips, **June–September 2023** (~200 MB). Larger dataset makes strategy differences more visible.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in range(6, 10):
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO, skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~50MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nTaxi data ready in MinIO!")

In [ ]:
taxi_df = spark.read.parquet(
    *[f"s3a://warehouse/raw/yellow_tripdata_2023-{m:02d}.parquet" for m in range(6, 10)]
)
taxi_df.cache()
taxi_df.createOrReplaceTempView("taxi_raw")
count = taxi_df.count()
print(f"Loaded {count:,} taxi trips (June-September 2023) for testing")
taxi_df.show(5)

## Part 1: Partition Granularity

### Strategy 1: Unpartitioned Table

In [ ]:
print("Creating UNPARTITIONED table...")
start = time.time()

taxi_df.writeTo("polaris.modeling.taxi_unpartitioned") \
    .using("iceberg") \
    .tableProperty("write.target-file-size-bytes", "134217728") \
    .createOrReplace()

write_time_unpart = time.time() - start
print(f"Write time: {write_time_unpart:.2f} seconds")

In [ ]:
print("Unpartitioned file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.modeling.taxi_unpartitioned.files
""").show()

### Strategy 2: Daily Partitioning

In [ ]:
print("Creating DAILY PARTITIONED table...")
start = time.time()

spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_daily")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_daily
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.target-file-size-bytes' = '134217728')
    AS SELECT * FROM taxi_raw
""")

write_time_daily = time.time() - start
print(f"Write time: {write_time_daily:.2f} seconds")

In [ ]:
print("Daily partition file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           COUNT(DISTINCT partition) as partition_count
    FROM polaris.modeling.taxi_daily.files
""").show()

### Strategy 3: Monthly Partitioning

In [ ]:
print("Creating MONTHLY PARTITIONED table...")
start = time.time()

spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_monthly")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_monthly
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES ('write.target-file-size-bytes' = '134217728')
    AS SELECT * FROM taxi_raw
""")

write_time_monthly = time.time() - start
print(f"Write time: {write_time_monthly:.2f} seconds")

In [ ]:
print("Monthly partition file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           COUNT(DISTINCT partition) as partition_count
    FROM polaris.modeling.taxi_monthly.files
""").show()

### Compare Query Performance

In [ ]:
query = """
    SELECT COUNT(*) as count, ROUND(AVG(fare_amount), 2) as avg_fare
    FROM {table}
    WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 00:00:00'
      AND tpep_pickup_datetime < TIMESTAMP '2023-06-16 00:00:00'
"""

print("Query: All trips on June 15, 2023")
print("=" * 60)

for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
                     ("Daily", "polaris.modeling.taxi_daily"),
                     ("Monthly", "polaris.modeling.taxi_monthly")]:
    start = time.time()
    result = spark.sql(query.format(table=table)).collect()
    elapsed = time.time() - start
    print(f"{name:15s}: {elapsed:.3f}s  (count: {result[0]['count']:,}, avg_fare: {result[0]['avg_fare']})")

In [ ]:
query2 = """
    SELECT COUNT(*) as count
    FROM {table}
    WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 14:00:00'
      AND tpep_pickup_datetime < TIMESTAMP '2023-06-15 15:00:00'
"""

print("\nQuery: Trips on June 15, 2-3 PM")
print("=" * 60)

for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
                     ("Daily", "polaris.modeling.taxi_daily"),
                     ("Monthly", "polaris.modeling.taxi_monthly")]:
    start = time.time()
    count = spark.sql(query2.format(table=table)).collect()[0]['count']
    elapsed = time.time() - start
    print(f"{name:15s}: {elapsed:.3f}s  (count: {count:,})")

### Partition Granularity Summary

In [ ]:
print("=" * 60)
print("PARTITION STRATEGY COMPARISON")
print("=" * 60)
print(f"{'Strategy':<20} {'Write Time':<12}")
print("-" * 40)
print(f"{'Unpartitioned':<20} {write_time_unpart:>10.2f}s")
print(f"{'Daily':<20} {write_time_daily:>10.2f}s")
print(f"{'Monthly':<20} {write_time_monthly:>10.2f}s")
print("=" * 60)

**Observations:**
- **Monthly** — fewest files, but day-range queries scan full month files
- **Daily** — good balance, one file/day, efficient for day-range queries
- **Unpartitioned** — simplest, but scans all data on any time filter

### Try It: Test with a Different Query Pattern

Above used a single-day filter. Try a full-month scan or a single-hour filter. How do strategies compare when the query aligns with partition granularity vs. when it doesn't?

In [ ]:
# my_query = """
#     SELECT COUNT(*), ROUND(AVG(total_amount), 2)
#     FROM {table}
#     WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 14:00:00'
#       AND tpep_pickup_datetime <  TIMESTAMP '2023-06-15 15:00:00'
# """
#
# for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
#                      ("Daily", "polaris.modeling.taxi_daily"),
#                      ("Monthly", "polaris.modeling.taxi_monthly")]:
#     start = time.time()
#     spark.sql(my_query.format(table=table)).collect()
#     elapsed = time.time() - start
#     print(f"{name:<20} {elapsed:.3f}s")

In [ ]:
for t in ['taxi_unpartitioned', 'taxi_daily', 'taxi_monthly']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Part 1 tables cleaned up!")

## Part 2: Sort Orders

### Table Without Sort Order

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_unsorted")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_unsorted
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM taxi_raw
""")

print("Unsorted table created!")

### Table With Sort Order

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_sorted")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_sorted
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM taxi_raw WHERE 1=0
""")

spark.sql("""
    ALTER TABLE polaris.modeling.taxi_sorted
    WRITE ORDERED BY PULocationID
""")

taxi_df.writeTo("polaris.modeling.taxi_sorted").append()

print("Sorted table created (sorted by PULocationID)!")

### Compare Query Performance

In [ ]:
query = """
    SELECT COUNT(*) as count, ROUND(AVG(fare_amount), 2) as avg_fare
    FROM {table}
    WHERE PULocationID = 132
      AND tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
"""

print("Query: Trips from location 132 on June 15")
print("=" * 60)

start = time.time()
r1 = spark.sql(query.format(table="polaris.modeling.taxi_unsorted")).collect()
unsorted_time = time.time() - start
print(f"Unsorted: {unsorted_time:.3f}s  (count: {r1[0]['count']}, avg_fare: {r1[0]['avg_fare']})")

start = time.time()
r2 = spark.sql(query.format(table="polaris.modeling.taxi_sorted")).collect()
sorted_time = time.time() - start
print(f"Sorted:   {sorted_time:.3f}s  (count: {r2[0]['count']}, avg_fare: {r2[0]['avg_fare']})")

if unsorted_time > sorted_time:
    print(f"\nSort order gave {unsorted_time/sorted_time:.1f}x speedup on filtered query")
else:
    print(f"\nResults similar at this data volume")

### Try It: Try a Different Sort Key

Above sorted on `PULocationID`. Try `fare_amount` or `trip_distance`. Filter on your sort key and compare to unsorted — does sort help more for high-selectivity filters?

In [ ]:
# my_sort_key = "fare_amount"  # or trip_distance, DOLocationID, etc.
# my_filter = "fare_amount > 100"  # match your sort key for best effect

# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_my_sort")
# spark.sql("""
#     CREATE TABLE polaris.modeling.taxi_my_sort
#     USING iceberg
#     PARTITIONED BY (days(tpep_pickup_datetime))
#     AS SELECT * FROM taxi_raw
#     WHERE 1=0
# """)
# spark.sql(f"ALTER TABLE polaris.modeling.taxi_my_sort WRITE ORDERED BY {my_sort_key}")
# taxi_df.writeTo("polaris.modeling.taxi_my_sort").append()
#
# start = time.time()
# spark.sql(f"SELECT COUNT(*) FROM polaris.modeling.taxi_my_sort WHERE {my_filter}").collect()
# print(f"Sorted by {my_sort_key}: {time.time() - start:.3f}s")
#
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_my_sort")

In [ ]:
for t in ['taxi_unsorted', 'taxi_sorted']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Part 2 tables cleaned up!")

## Part 3: Distribution Modes

A Spark-specific Iceberg setting controlling whether/how Spark **shuffles** data across tasks before writing. Shuffling sends rows across the network so all rows for one partition land on one task — costs network I/O but produces cleaner files. Three modes:

- **`hash`** — shuffle by partition value; each task writes one partition. Default for tables **without** a sort order.
- **`range`** — range shuffle that groups by partition **and** sorts within it. Default for tables **with** a sort order.
- **`none`** — no shuffle. Faster if data is already organized; risky for small files otherwise.

Whether mode matters depends on the relationship between your data's natural ordering and the partition key. Two experiments follow — first verify the raw data is naturally ordered by pickup date.

### Raw Data is Already Ordered by Day

NYC taxi Parquet files ship one per month, rows in chronological order → naturally clustered by day with no shuffle. Verify below.

In [ ]:
from pyspark.sql.functions import to_date, min as spark_min, max as spark_max

print("Date range per source parquet file:")
for m in range(6, 10):
    path = f"s3a://warehouse/raw/yellow_tripdata_2023-{m:02d}.parquet"
    stats = spark.read.parquet(path) \
        .filter("tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-10-01'") \
        .agg(
            spark_min(to_date("tpep_pickup_datetime")).alias("first"),
            spark_max(to_date("tpep_pickup_datetime")).alias("last"),
            F.count("*").alias("n")
        ).collect()[0]
    print(f"  {path.split('/')[-1]}: {stats.first} to {stats.last}  ({stats.n:,} rows)")

In [ ]:
print("Trips per day across the dataset (first and last 5 days shown):")
print("The data follows a smooth chronological progression.\n")

daily = taxi_df \
    .filter("tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-10-01'") \
    .withColumn("pickup_date", to_date("tpep_pickup_datetime")) \
    .groupBy("pickup_date") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy("pickup_date")

total_days = daily.count()
print(f"Total distinct days: {total_days}\n")

print("First 5 days:")
daily.limit(5).show(truncate=False)
print("Last 5 days:")
daily.orderBy(F.desc("pickup_date")).limit(5).orderBy("pickup_date").show(truncate=False)

print("The data spans a contiguous range of dates, confirming the parquet files")
print("are organized chronologically. This natural ordering matters for distribution modes.")

Each input partition spans a few days, in chronological order. This drives distribution-mode behavior: when data is already grouped by the partition key, the shuffle is unnecessary; otherwise it's essential.

Two experiments:
- **A:** `bucket(16, PULocationID)` — data **not** sorted by this col → distribution mode matters a lot.
- **B:** `days(tpep_pickup_datetime)` — data **is** sorted by day → `hash` and `none` produce nearly identical results.

### Experiment A: Bucket Partitioning (Data Not Pre-Sorted)

`bucket(N, col)` hashes into N groups — useful for high-cardinality columns where temporal transforms don't apply. Pick N from your target file count; more buckets = more files but finer pruning.

When partitioning on a column the data isn't organized by, distribution mode controls the pre-write shuffle. Compare all three.

#### Hash Distribution (default, no sort order)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_hash")

print("Writing with HASH distribution mode (bucketed by PULocationID)...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_hash
    USING iceberg
    PARTITIONED BY (bucket(16, PULocationID))
    TBLPROPERTIES ('write.distribution-mode' = 'hash')
    AS SELECT * FROM taxi_raw
""")

hash_time = time.time() - start
print(f"Write time: {hash_time:.2f} seconds")

In [ ]:
print("Hash distribution file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.modeling.taxi_dist_hash.files
""").show()

#### Range Distribution (default with sort order)

When a table has a sort order, Iceberg's Spark integration auto-uses **range** distribution: samples data to estimate value distributions and partition boundaries (small overhead vs hash), then distributes so each task writes one partition with data already sorted → clean partitioning + sorted files in one pass.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_range")

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_range
    USING iceberg
    PARTITIONED BY (bucket(16, PULocationID))
    AS SELECT * FROM taxi_raw WHERE 1=0
""")

spark.sql("""
    ALTER TABLE polaris.modeling.taxi_dist_range
    WRITE ORDERED BY fare_amount
""")

print("Writing with RANGE distribution mode (bucketed + sorted by fare_amount)...")
start = time.time()

taxi_df.writeTo("polaris.modeling.taxi_dist_range").append()

range_time = time.time() - start
print(f"Write time: {range_time:.2f} seconds")

In [ ]:
print("Range distribution file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.modeling.taxi_dist_range.files
""").show()

#### None Distribution (no shuffle)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_none")

print("Writing with NONE distribution mode (bucketed by PULocationID)...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_none
    USING iceberg
    PARTITIONED BY (bucket(16, PULocationID))
    TBLPROPERTIES ('write.distribution-mode' = 'none')
    AS SELECT * FROM taxi_raw
""")

none_time = time.time() - start
print(f"Write time: {none_time:.2f} seconds")

In [ ]:
print("None distribution file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb
    FROM polaris.modeling.taxi_dist_none.files
""").show()

#### Experiment A Summary

In [ ]:
hash_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_hash.files").collect()[0].f
range_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_range.files").collect()[0].f
none_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_none.files").collect()[0].f

print("=" * 80)
print("EXPERIMENT A: bucket(16, PULocationID), data NOT pre-sorted")
print("=" * 80)
print(f"{'Mode':<12} {'Sort Order':<14} {'Time':<8} {'Files':<8} {'Files/Partition':<18} {'Notes'}")
print("-" * 80)
print(f"{'Hash':<12} {'--':<14} {hash_time:>5.1f}s  {hash_stats:>5}   {hash_stats//16:>5}              {'Clean layout'}")
range_fpp = f"~{range_stats // 16}"
print(f"{'Range':<12} {'fare_amount':<14} {range_time:>5.1f}s  {range_stats:>5}   {range_fpp:>5}              {'Clean + sorted'}")
print(f"{'None':<12} {'--':<14} {none_time:>5.1f}s  {none_stats:>5}   {none_stats//16:>5}              {'Small file problem!'}")
print("=" * 80)

When data is **not** pre-sorted by the partition key, distribution mode matters a lot:

- **Hash** — shuffle by partition value → each task writes one bucket → clean uniform files.
- **Range** — also groups by partition, plus sorts within. Slightly slower, maybe a few extra files from size splits, but sorted layout helps read-time queries on the sort column.
- **None** — no shuffle → each task writes to every bucket it touches. With `PULocationID` scattered randomly → 8 files per bucket: textbook small-file problem.

### Experiment B: Day Partitioning (Data Already Pre-Sorted)

Raw data is already chronological. Partition by `days(tpep_pickup_datetime)` → data naturally groups by partition key. Does the shuffle still help? Compare `hash` vs `none` with daily partitioning.

#### Hash Distribution (daily)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_hash_day")

print("Writing with HASH distribution mode (daily partition)...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_hash_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.distribution-mode' = 'hash')
    AS SELECT * FROM taxi_raw
""")

hash_day_time = time.time() - start
print(f"Write time: {hash_day_time:.2f} seconds")

In [ ]:
print("Hash distribution (daily) file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb
    FROM polaris.modeling.taxi_dist_hash_day.files
""").show()

#### None Distribution (daily)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_none_day")

print("Writing with NONE distribution mode (daily partition, data is pre-sorted)...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_none_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.distribution-mode' = 'none')
    AS SELECT * FROM taxi_raw
""")

none_day_time = time.time() - start
print(f"Write time: {none_day_time:.2f} seconds")

In [ ]:
print("None distribution (daily, pre-sorted) file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb
    FROM polaris.modeling.taxi_dist_none_day.files
""").show()

#### Experiment B Summary

In [ ]:
hash_day_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_hash_day.files").collect()[0].f
noneday_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_none_day.files").collect()[0].f
hash_day_parts = spark.sql("SELECT COUNT(DISTINCT partition) as p FROM polaris.modeling.taxi_dist_hash_day.files").collect()[0].p
noneday_parts = spark.sql("SELECT COUNT(DISTINCT partition) as p FROM polaris.modeling.taxi_dist_none_day.files").collect()[0].p

print("=" * 70)
print("EXPERIMENT B: days(tpep_pickup_datetime), data IS pre-sorted")
print("=" * 70)
print(f"{'Mode':<12} {'Time':<8} {'Files':<8} {'Partitions':<12} {'Files/Partition':<18}")
print("-" * 70)
hash_day_fpp = f"{hash_day_stats / hash_day_parts:.1f}"
none_day_fpp = f"{noneday_stats / noneday_parts:.1f}"
print(f"{'Hash':<12} {hash_day_time:>5.1f}s  {hash_day_stats:>5}   {hash_day_parts:>8}     {hash_day_fpp:>5}")
print(f"{'None':<12} {none_day_time:>5.1f}s  {noneday_stats:>5}   {noneday_parts:>8}     {none_day_fpp:>5}")
print("=" * 70)


When data **is** ordered by the partition key, both `hash` and `none` produce well-partitioned files (no partition mixing in either). `none` makes more files (200 vs 129) because boundary partitions split across Spark tasks — each task spanning a day boundary writes a small fragment to each day. But contents are well-partitioned → similar query performance.

**Bottom line:** check if your data's natural ordering matches the partition key. If yes, `none` skips unnecessary shuffle overhead (maybe a few extra boundary files). If no (Experiment A), `hash`/`range` is essential.

### Sort Order Degradation Over Multiple Writes

A fresh sorted table has non-overlapping file ranges (e.g. sorted by `fare_amount`: file 1 = $0–15, file 2 = $15–30…). A query `WHERE fare_amount BETWEEN 50 AND 55` skips most files via min/max metadata.

But each new write (INSERT/MERGE) creates its **own** sorted files covering the **full** range, overlapping earlier writes. After many writes, every range has multiple overlapping files → min/max pruning degrades.

Demonstration: insert the same month repeatedly and measure as overlaps accumulate.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_sort_decay")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_sort_decay
    USING iceberg
    TBLPROPERTIES ('write.target-file-size-bytes' = '8388608')
    AS SELECT * FROM taxi_raw WHERE 1=0
""")
spark.sql("ALTER TABLE polaris.modeling.taxi_sort_decay WRITE ORDERED BY fare_amount")

june_df = taxi_df.filter("tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-07-01'")
june_df.createOrReplaceTempView("june_trips")
june_count = june_df.count()
print(f"Created unpartitioned table sorted by fare_amount (8 MB target file size)")
print(f"Using June data ({june_count:,} rows per write). We'll insert it repeatedly")

In [ ]:
benchmark_query = """
    SELECT COUNT(*), ROUND(AVG(tip_amount), 2)
    FROM polaris.modeling.taxi_sort_decay
    WHERE fare_amount BETWEEN 50 AND 55
"""

overlap_query = """
    SELECT COUNT(*) FROM polaris.modeling.taxi_sort_decay.files
    WHERE readable_metrics.fare_amount.lower_bound <= 55
      AND readable_metrics.fare_amount.upper_bound >= 50
"""

def write_and_benchmark(label):
    file_count = spark.sql(
        "SELECT COUNT(*) FROM polaris.modeling.taxi_sort_decay.files"
    ).collect()[0][0]
    overlap = spark.sql(overlap_query).collect()[0][0]
    t = time.time()
    spark.sql(benchmark_query).collect()
    elapsed = time.time() - t
    print(f"{label:<22} {file_count:>6}      {overlap:>6} / {file_count:<6}    {elapsed:.3f}s")
    return file_count, overlap, elapsed

checkpoints = [1, 10]
current_writes = 0

print("Inserting June data repeatedly and benchmarking...\n")
print(f"{'State':<22} {'Files':<10} {'Matching $50-$55':<18} {'Query Time'}")
print("-" * 68)

for target in checkpoints:
    while current_writes < target:
        spark.sql("INSERT INTO polaris.modeling.taxi_sort_decay SELECT * FROM june_trips")
        current_writes += 1
    write_and_benchmark(f"After {current_writes} writes")

Each of the 10 writes made its own sorted files. Every batch has a file covering $50–55, one covering $10–15, etc. → 10 files now overlap any `fare_amount` query, one per batch.

Now compact all 10 batches into one globally sorted set via `rewrite_data_files`.

In [ ]:
print("Compacting with sort order...")
start = time.time()

result = spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'modeling.taxi_sort_decay',
        strategy => 'sort'
    )
""").collect()[0]

compact_time = time.time() - start
print(f"Compaction took {compact_time:.1f}s")
print(f"Rewrote {result['rewritten_data_files_count']} files into {result['added_data_files_count']} files")
print()
write_and_benchmark("After compaction")


**Why:** each INSERT makes independently sorted files. After write 1: one file per range. After 10 writes: 10 files per range. A `fare_amount` filter must scan all of them — min/max pruning can't eliminate overlapping files.

`rewrite_data_files` with `strategy => 'sort'` re-sorts files per the table's sort order → one set of non-overlapping ranges. Default `'binpack'` only merges small files without re-sorting (faster, no sort benefit) — that's what E3.2 did. After sort compaction, queries scan only files overlapping the predicate. Similar to `OPTIMIZE` in other formats.

Production: schedule periodic `rewrite_data_files` (nightly / after N ingestion cycles) to consolidate overlapping files and keep sort benefits.

### Try It: Combine Strategies

Build a table combining partitioning + sort order + distribution mode — e.g. daily partition + sorted by `PULocationID` + hash distribution. Write data and compare query performance against earlier tables. What combo works best for your query pattern?

In [ ]:
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_combined")
# spark.sql("""
#     CREATE TABLE polaris.modeling.taxi_combined
#     USING iceberg
#     PARTITIONED BY (days(tpep_pickup_datetime))
#     TBLPROPERTIES ('write.distribution-mode' = 'hash')
#     AS SELECT * FROM taxi_raw
#     WHERE 1=0
# """)
# spark.sql("ALTER TABLE polaris.modeling.taxi_combined WRITE ORDERED BY PULocationID")
# taxi_df.writeTo("polaris.modeling.taxi_combined").append()
#
# start = time.time()
# spark.sql("""
#     SELECT COUNT(*), ROUND(AVG(fare_amount), 2)
#     FROM polaris.modeling.taxi_combined
#     WHERE PULocationID = 132
#       AND tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
# """).collect()
# print(f"Combined strategy: {time.time() - start:.3f}s")
#
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_combined")

## Cleanup

In [ ]:
taxi_df.unpersist()
for t in ['taxi_dist_hash', 'taxi_dist_range', 'taxi_dist_none',
          'taxi_dist_hash_day', 'taxi_dist_none_day', 'taxi_sort_decay']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Cleanup complete!")